# 02 Cleaning
This notebook cleans the raw smartwatch health data. We log every transformation step and export to .

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
RAW_PATH = Path('../data/raw/unclean_smartwatch_health_data.csv')
PROCESSED_PATH = Path('../data/processed/clean_smartwatch_health_data.csv')

# Load the raw data
df = pd.read_csv(RAW_PATH)
df.info()

In [ ]:
# Transformation 1: Normalize column names
df.columns = df.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
print('New columns:', df.columns.tolist())

In [ ]:
# Transformation 2: Handle duplicates and missing User IDs
df = df.drop_duplicates()
df = df.dropna(subset=['user_id'])
print(f'Rows after deduplication and dropping null user_id: {len(df)}')

In [ ]:
# Transformation 3: Clean 'activity_level'
# Fix misspellings and standardize categories
df['activity_level'] = df['activity_level'].astype(str).str.strip().str.lower()
df['activity_level'] = df['activity_level'].replace({
    'highly_active': 'highly active',
    'actve': 'active',
    'seddentary': 'sedentary',
    'nan': np.nan
})
print(df['activity_level'].value_counts(dropna=False))

In [ ]:
# Transformation 4: Clean 'sleep_duration_hours'
# Convert to numeric, pushing 'ERROR' or invalid values to NaN
df['sleep_duration_hours'] = pd.to_numeric(df['sleep_duration_hours'], errors='coerce')

In [ ]:
# Transformation 5: Clean 'stress_level'
# Convert to numeric, pushing 'Very High' or invalid strings to NaN
df['stress_level'] = pd.to_numeric(df['stress_level'], errors='coerce')

In [ ]:
# Transformation 6: Cast 'step_count' and 'user_id' to integer
df['step_count'] = df['step_count'].round().astype('Int64')
df['user_id'] = df['user_id'].round().astype('Int64')

In [ ]:
# Final check
df.info()
df.head()

In [ ]:
# Export cleaned data
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Cleaned dataset saved to {PROCESSED_PATH}')